In [1]:
import pickle
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import ( Input, Embedding, GRU, Dense )
from tensorflow.keras.callbacks import EarlyStopping
from keras.callbacks import ReduceLROnPlateau

In [2]:
with open( "data/rewriting_train.pkl", "rb" ) as f: encoder_train, decoder_input_train, decoder_target_train = pickle.load(f)

In [3]:
with open( "data/rewriting_val.pkl", "rb" ) as f: encoder_val, decoder_input_val, decoder_target_val = pickle.load(f)

In [4]:
with open( "data/rewriting_sequence_info.pkl", "rb" ) as f: sequence_info = pickle.load(f)

In [5]:
source_vocab_size = sequence_info["source_vocab_size"]
target_vocab_size = sequence_info["target_vocab_size"]

max_source_length = sequence_info["max_source_length"]
max_target_length = sequence_info["max_target_length"]

In [6]:
print("Source vocabulary:", source_vocab_size)
print("Target vocabulary:", target_vocab_size) 
print("Source sequence length:", max_source_length) 
print("Target sequence length:", max_target_length)

Source vocabulary: 10320
Target vocabulary: 11706
Source sequence length: 17
Target sequence length: 19


In [7]:
embedding_dim = 128 
latent_dim = 256

In [8]:
encoder_inputs = Input( shape=(max_source_length,), name="encoder_inputs" )

In [9]:
encoder_embedding = Embedding(input_dim=source_vocab_size, output_dim=embedding_dim, mask_zero=True, name="encoder_embedding" )(encoder_inputs)

In [10]:
encoder_gru = GRU( latent_dim, return_state=True, name="encoder_gru" )

In [11]:
encoder_outputs, encoder_state = encoder_gru( encoder_embedding )

In [12]:
decoder_inputs = Input( shape=(max_target_length - 1,), name="decoder_inputs" )

In [13]:
decoder_embedding = Embedding( input_dim=target_vocab_size, output_dim=embedding_dim, mask_zero=True, name="decoder_embedding" )(decoder_inputs)

In [14]:
decoder_gru = GRU( latent_dim, return_sequences=True, return_state=True, name="decoder_gru" )

In [15]:
decoder_outputs, _ = decoder_gru( decoder_embedding, initial_state=encoder_state )

In [16]:
decoder_dense = Dense( target_vocab_size, activation="softmax", name="decoder_output" )

In [17]:
decoder_outputs = decoder_dense( decoder_outputs )

In [18]:
model = Model( [encoder_inputs, decoder_inputs], decoder_outputs )

In [19]:
model.compile( optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"] )

In [20]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,
    min_delta=0.001,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [22]:
history = model.fit(
    [encoder_train, decoder_input_train],
    decoder_target_train,
    validation_data=(
        [encoder_val, decoder_input_val],
        decoder_target_val
    ),
    epochs=30,
    batch_size=64,
    callbacks=[
        early_stopping,
        reduce_lr
    ]
)

Epoch 1/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 50s 338ms/step - accuracy: 0.2724 - loss: 4.7385 - val_accuracy: 0.2622 - val_loss: 5.4382 - learning_rate: 5.0000e-04
Epoch 2/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 50s 337ms/step - accuracy: 0.2812 - loss: 4.5906 - val_accuracy: 0.2661 - val_loss: 5.4213 - learning_rate: 5.0000e-04
Epoch 3/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 50s 338ms/step - accuracy: 0.2887 - loss: 4.4540 - val_accuracy: 0.2696 - val_loss: 5.4062 - learning_rate: 5.0000e-04
Epoch 4/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 50s 339ms/step - accuracy: 0.2966 - loss: 4.3195 - val_accuracy: 0.2738 - val_loss: 5.3905 - learning_rate: 5.0000e-04
Epoch 5/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 50s 338ms/step - accuracy: 0.3040 - loss: 4.1898 - val_accuracy: 0.2756 - val_loss: 5.4107 - learning_rate: 5.0000e-04
Epoch 6/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 51s 344ms/step - accuracy: 0.3114 - loss: 4.0630 - val_accuracy: 0.2830 - val_loss: 5.4035 - learning_rate: 5.0000e-04
Epoch 7/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 325

In [23]:
model.save( "models/rewriting_seq2seq_gru.keras" )